# Phase 4: matched GPU pilot

Select **Runtime > Change runtime type > GPU** and obtain a GPU before starting. This thin notebook prepares the reviewed source and calls its pilot CLI; it contains no model implementation. Joint TAR requires a real CUDA device. No GPU job has been executed merely because this notebook exists.

Set `PROJECT_SHA` to the full reviewed and pushed commit. The placeholder fails before mounting Drive or downloading anything. Mount your private Drive interactively; all console logs and raw run outputs persist there. Leave `LAUNCH_PILOT = False` while checking setup. Set it to `True` only when ready to execute the preregistered GPU pilot. Do not change training settings after viewing pilot outcomes.

See [GPU workflow](https://github.com/diyarh26/residual-guided-text-tabular/blob/main/docs/PHASE4_GPU.md) for prerequisites and interruption limitations. A fixed historical TAR batch of 256 can exhaust device memory; record failure rather than silently changing it.


In [ ]:
import re
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

PROJECT_SHA = "REPLACE_WITH_FULL_40_CHARACTER_PROJECT_SHA"
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_SHA) is None:
    raise ValueError("Set PROJECT_SHA to the full lowercase 40-character reviewed and pushed ReFuTab commit.")

REPOSITORY = "https://github.com/diyarh26/residual-guided-text-tabular.git"
PROJECT_DIR = Path("/content/refutab")
RUNS_ROOT = Path("/content/drive/MyDrive/ReFuTab/runs")
RESUME_RUN = None  # Existing private Drive run directory, or None for a new pilot.
RUN_ID = "phase4-colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
print("Project commit:", PROJECT_SHA)
print("Run ID:", RUN_ID)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_LOG = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".log")
HARDWARE_RECORD = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".hardware.json")
BOOTSTRAP_LOG.parent.mkdir(parents=True, exist_ok=True)
with BOOTSTRAP_LOG.open("x", encoding="utf-8") as stream:
    stream.write(f"project_sha={PROJECT_SHA}\nrun_id={RUN_ID}\nrequested_hardware=cuda\nservice_cost=unmeasured\n")
print("Persistent console log:", BOOTSTRAP_LOG)


In [ ]:
import os
import shlex
import subprocess
import sys

def run(command, *, cwd=None):
    # Each command streams to Drive from startup; failures stop the cell.
    lines = []
    with BOOTSTRAP_LOG.open("a", encoding="utf-8") as stream:
        heading = "$ " + shlex.join([str(part) for part in command])
        print(heading)
        stream.write(heading + "\n")
        stream.flush()
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                stream.write(line)
                stream.flush()
                lines.append(line)
            return_code = process.wait()
        stream.write(f"exit_code={return_code}\n")
        stream.flush()
    if return_code:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {BOOTSTRAP_LOG}")
    return "".join(lines)

if PROJECT_DIR.exists():
    raise FileExistsError("/content/refutab already exists. Use a fresh runtime or explicitly move that checkout before retrying.")
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.11.24"])
run(["git", "clone", REPOSITORY, str(PROJECT_DIR)])
run(["git", "checkout", "--detach", PROJECT_SHA], cwd=PROJECT_DIR)
if run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR).strip() != PROJECT_SHA:
    raise RuntimeError("Source commit mismatch.")
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Source checkout must be clean before setup.")
run(["uv", "sync", "--project", "environments/gpu", "--locked", "--python", "3.11"], cwd=PROJECT_DIR)
UV_PYTHON = ["uv", "run", "--project", "environments/gpu", "--frozen", "python"]


## Verify the actual GPU before preparing inputs

The separate Linux/Python 3.11 environment pins Torch 2.7.1+cu126 and torchvision 0.22.1+cu126; it does not replace the repository's CPU environment. This check records the actual GPU, driver, visible memory, host RAM and package build, and executes CUDA work. A successful check establishes device availability, not sufficient memory for TAR or a successful training result.


In [ ]:
run(["nvidia-smi"], cwd=PROJECT_DIR)
# Upstream TAR checks this variable. Its value selects a real verified device.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU_PREFLIGHT = r"""
import json
import platform
import subprocess
import sys
from pathlib import Path
import psutil
import torch
import torchvision

if sys.platform != "linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 11):
    raise RuntimeError("GPU environment requires Linux x86_64 and Python 3.11.")
if torch.__version__ != "2.7.1+cu126" or torchvision.__version__ != "0.22.1+cu126":
    raise RuntimeError("Unexpected Torch/torchvision build; restore the committed GPU lock.")
if torch.version.cuda != "12.6" or not torch.cuda.is_available():
    raise RuntimeError("A working NVIDIA GPU and compatible driver are required; CPU fallback is disabled.")
torch.cuda.set_device(0)
x = torch.ones((32, 32), device="cuda")
if float((x @ x)[0, 0].item()) != 32.0:
    raise RuntimeError("CUDA matrix multiplication failed.")
torch.cuda.synchronize()
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
record = {
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda, "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_name": properties.name, "cuda_compute_capability": [properties.major, properties.minor],
    "cuda_total_memory_bytes": total_bytes, "cuda_free_memory_bytes": free_bytes,
    "host_total_memory_bytes": psutil.virtual_memory().total,
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
    "cuda_matmul_passed": True, "training_executed": False, "service_cost": "unmeasured"
}
with Path(sys.argv[1]).open("x", encoding="utf-8") as stream:
    json.dump(record, stream, indent=2, allow_nan=False)
    stream.write("\n")
print(json.dumps(record, indent=2, allow_nan=False))
"""
run(UV_PYTHON + ["-c", GPU_PREFLIGHT, str(HARDWARE_RECORD)], cwd=PROJECT_DIR)


In [ ]:
run(UV_PYTHON + ["scripts/prepare_phase1.py"], cwd=PROJECT_DIR)
run(UV_PYTHON + ["scripts/prepare_phase4.py"], cwd=PROJECT_DIR)
run(UV_PYTHON + ["-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_DIR)
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Preparation/tests modified source or created unignored files; inspect before fitting.")
print("Preparation, device preflight and tests passed. Training has not started.")


## Launch the preregistered pilot explicitly

Review the committed config, cost, actual GPU and free memory first. Set `LAUNCH_PILOT = True` in the launch cell below and run that cell when ready. Do not rerun the settings cell merely to launch; it generates a new run ID. All raw models, predictions, splits and manifests remain private in Drive. Do not publish those files by committing them.


In [ ]:
LAUNCH_PILOT = False
if not LAUNCH_PILOT:
    raise RuntimeError("GPU pilot is prepared. Set LAUNCH_PILOT = True explicitly when ready to run.")
if run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR).strip() != PROJECT_SHA:
    raise RuntimeError("Source commit changed before launch.")
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Source checkout must be clean before training.")
pilot_command = UV_PYTHON + ["scripts/run_pilot.py", "--config", "configs/phase4_pilot.json"]
if RESUME_RUN is None:
    pilot_command += ["--run-id", RUN_ID, "--output-root", str(RUNS_ROOT)]
    pilot_directory = RUNS_ROOT / RUN_ID
else:
    pilot_directory = Path(RESUME_RUN).resolve()
    if not pilot_directory.is_dir() or RUNS_ROOT.resolve() not in pilot_directory.parents:
        raise ValueError("RESUME_RUN must identify an existing private run directory inside RUNS_ROOT.")
    pilot_command += ["--resume-run", str(pilot_directory)]
run(pilot_command, cwd=PROJECT_DIR)
print("Review status, comparisons and artifacts under:", pilot_directory)
print("Persistent console log:", BOOTSTRAP_LOG)
print("Actual hardware preflight:", HARDWARE_RECORD)


## After execution or interruption

Inspect the saved manifest and every method's status. A finished command, available checkpoint or browser disconnection is insufficient evidence that all pilot conditions completed. Record actual runtime, hardware, cost and any failures. Report the pilot recommendation before beginning full evaluation.

Keep interrupted attempts and their bootstrap logs. To continue the same ledger, set `RESUME_RUN` to its existing private Drive directory before setup. The CLI validates and reuses completed stages, creates a new attempt for interrupted work, and preserves old attempts. Follow the documented `run_pilot.py` resumption contract in [GPU workflow](https://github.com/diyarh26/residual-guided-text-tabular/blob/main/docs/PHASE4_GPU.md); do not overwrite completed or failed artifacts. TAR resumes only validated Trainer checkpoints, including optimizer and callback state, or restores a completed selected backbone without extra training before PCA. This path has CPU fixture coverage; actual CUDA continuation is unvalidated. LightGBM fitting has no optimizer continuation and restarts in a new attempt if interrupted. Export only reviewed aggregate summaries to GitHub. This notebook has not been executed in Colab until an actual execution is recorded.
